# 03 · Structured extraction + PSX test set
**Component:** extract revenue, profit before tax, profit after tax, EPS, total assets and total equity from a PSX report into a validated schema.
**Alternatives:** `rules` (regex, no AI) vs `api` (Gemini, JSON mode + Pydantic) vs `local` (Qwen2.5-7B via Ollama).
**Test set (ours):** 10 PSX annual reports × 6 fields = 60 hand-labelled examples. The same labels also produce a PSX QA set: 60 look-ups plus about 30 growth questions, with gold answers computed in code.

### Labelling protocol (about 2 hours for two people)
1. Put the PDF links in `data/psx_reports.csv` (from dps.psx.com.pk/company/SYMBOL → Financial Reports).
2. `python fetch_psx_reports.py`
3. Fill `data/psx_labels.csv`: copy values exactly as printed (brackets = negative), plus the unit and the 1-based PDF page. Use the **consolidated** statements if the report has both.
4. Each member labels 5 reports, then cross-checks 6 rows of the other's work. Record any disagreements in the `notes` column.

In [ ]:
# --- Setup (works on Google Colab and locally) ---------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("mizan-ai"):
    subprocess.run(["git", "clone", "https://github.com/usmanbari-ub/mizan-ai.git"], check=True)
ROOT = os.path.abspath("mizan-ai" if IN_COLAB else "../..")
os.chdir(os.path.join(ROOT, "eval"))
sys.path[:0] = [os.path.join(ROOT, "eval"), os.path.join(ROOT, "backend")]
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(ROOT, "backend/requirements.txt")], check=True)
print("working dir:", os.getcwd())

In [ ]:
!python fetch_psx_reports.py
!python build_psx_testset.py

In [ ]:
import os
os.environ.setdefault("MIZAN_LLM_API_KEY", "PASTE_YOUR_KEY")
# for the local model: install Ollama, `ollama pull qwen2.5:7b`, then add "local" below
!python run_extraction.py --extractors rules api

In [ ]:
import json, pandas as pd
ex = json.load(open("results/summary.json"))["extraction"]
display(pd.DataFrame(ex).T)
errs = pd.DataFrame(json.load(open("results/extraction_errors.json")))
display(errs.groupby(["extractor","cause"]).size().rename("count").to_frame())
errs.groupby("cause").head(3)

## PSX QA (unseen test set: re-checks the retrieval and QA results on Pakistani filings)

In [ ]:
!python run_qa.py --dataset psx --modes rag_direct rag_calc agent --runs 1 --retriever bm25x
print(json.dumps(json.load(open("results/summary.json")).get("qa_psx", {}), indent=1)[:3000])